<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/InferentialDS1_extracted_NN1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CaO content Inferential DS1
Working on Extracted Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

sns.set()
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
sklearn.__version__

'1.6.1'

Read from Google Drive

In [27]:
from google.colab import files
import pandas as pd


uploaded = files.upload()


#df = pd.read_csv('df_clean_avg.csv', sep='\t')   # <-- plik oddzielany tabulatorem
df_raw = pd.read_csv('df_clean_avg.csv', sep=';')   # <-- plik oddzielany przecinkiem
print("Data loaded correctly!\n")
print(df_raw.head())

Saving df_clean_avg.csv to df_clean_avg (1).csv
Data loaded correctly!

            ChangeTime  LabUpdate Inferential_avg Inferential_med  \
0  2025-10-16 03:02:30         68       52,716789     51,08698835   
1  2025-10-16 06:33:30         60     62,41755307     63,74654023   
2  2025-10-16 08:32:30         31     44,39371878     46,67294865   
3  2025-10-16 10:35:40         32     23,31702046     23,99194305   
4  2025-10-16 12:45:30         31      20,6122777     21,00740895   

     Ratio_avg    Ratio_med       pH_avg       pH_med   CMFlow_avg  \
0  0,380589473  0,380293399  9,113029238  9,113762856  52,38471108   
1   0,35585871  0,356621683  9,092202058  9,092871666   62,5942022   
2  0,342781166  0,342676342  9,090918157  9,092871666  45,91751767   
3  0,357974786  0,357905269  9,116514277  9,113762856  24,13672841   
4    0,3405839  0,338620573  9,017560911  9,018521309  21,07282445   

    CMFlow_med     error_avg     error_med  
0  51,07588959    -15,283211  -16,91301165  
1 

Podstawowe Informacje

In [28]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ChangeTime       66 non-null     object
 1   LabUpdate        66 non-null     int64 
 2   Inferential_avg  66 non-null     object
 3   Inferential_med  66 non-null     object
 4   Ratio_avg        66 non-null     object
 5   Ratio_med        66 non-null     object
 6   pH_avg           66 non-null     object
 7   pH_med           66 non-null     object
 8   CMFlow_avg       66 non-null     object
 9   CMFlow_med       66 non-null     object
 10  error_avg        66 non-null     object
 11  error_med        66 non-null     object
dtypes: int64(1), object(11)
memory usage: 6.3+ KB


In [29]:
dataset = df_raw.copy()
dataset.head()

,ChangeTime,LabUpdate,Inferential_avg,Inferential_med,Ratio_avg,Ratio_med,pH_avg,pH_med,CMFlow_avg,CMFlow_med,error_avg,error_med
0,2025-10-16 03:02:30,68,"52,716789","51,08698835","0,380589473","0,380293399","9,113029238","9,113762856","52,38471108","51,07588959","-15,283211","-16,91301165"
1,2025-10-16 06:33:30,60,"62,41755307","63,74654023","0,35585871","0,356621683","9,092202058","9,092871666","62,5942022","64,00079346","2,417553072","3,746540225"
2,2025-10-16 08:32:30,31,"44,39371878","46,67294865","0,342781166","0,342676342","9,090918157","9,092871666","45,91751767","46,83626556","13,39371878","15,67294865"
3,2025-10-16 10:35:40,32,"23,31702046","23,99194305","0,357974786","0,357905269","9,116514277","9,113762856","24,13672841","24,36791992","-8,682979535","-8,008056952"
4,2025-10-16 12:45:30,31,"20,6122777","21,00740895","0,3405839","0,338620573","9,017560911","9,018521309","21,07282445","21,40418625","-10,3877223","-9,992591051"


In [30]:
dataset = dataset.replace(',', '.', regex=True)
dataset = dataset.apply(pd.to_numeric, errors='ignore')


/tmp/ipython-input-3462246338.py:2: FutureWarning:

errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead



In [31]:
dataset = dataset.drop(columns=['ChangeTime', 'Inferential_avg', 'Inferential_med', 'Ratio_med','pH_med','CMFlow_avg','CMFlow_med','error_avg','error_med'])
dataset.head()

,LabUpdate,Ratio_avg,pH_avg
0,68,0.380589,9.113029
1,60,0.355859,9.092202
2,31,0.342781,9.090918
3,32,0.357975,9.116514
4,31,0.340584,9.017561


In [32]:
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

print(f'train_dataset length: {len(train_dataset)}')
print(f'test_dataset length: {len(test_dataset)}')

train_dataset length: 53
test_dataset length: 13


In [33]:
train_dataset.head()

,LabUpdate,Ratio_avg,pH_avg
45,24,0.353721,8.939285
28,34,0.369435,9.086791
29,47,0.365395,9.103895
55,36,0.361612,8.797188
63,51,0.387047,8.796699


In [34]:
test_dataset.head()

,LabUpdate,Ratio_avg,pH_avg
0,68,0.380589,9.113029
3,32,0.357975,9.116514
9,30,0.392138,9.099190
19,48,0.390975,8.989384
21,45,0.400103,8.990417


In [35]:
%tensorflow_version 2.x
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.set_printoptions(precision=12, suppress=True, linewidth=150)
pd.options.display.float_format = '{:.6f}'.format
tf.__version__
px.scatter_matrix(train_dataset, dimensions=['LabUpdate', 'Ratio_avg', 'pH_avg'], color='LabUpdate', height=700)

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


In [36]:
train_stats = train_dataset.describe()
train_stats.pop('LabUpdate')
train_stats = train_stats.transpose()
train_stats

,count,mean,std,min,25%,50%,75%,max
Ratio_avg,53.000000,0.368652,0.021684,0.306806,0.355859,0.365015,0.378592,0.438227
pH_avg,53.000000,8.974215,0.121867,8.596300,8.868478,9.024020,9.075794,9.108848


In [37]:
train_labels = train_dataset.pop('LabUpdate')
test_labels = test_dataset.pop('LabUpdate')

In [38]:
def norm(x):
    return (x - train_stats['mean']) / train_stats['std']

In [39]:
normed_train_data = norm(train_dataset)
normed_test_data = norm(test_dataset)

In [40]:
normed_train_data.isnull().sum()

,0
Ratio_avg,0
pH_avg,0


In [44]:
normed_test_data = normed_test_data
normed_train_data = normed_train_data

In [45]:
def norm(x):
    return (x - train_stats['mean']) / train_stats['std']

In [46]:
normed_train_data = norm(train_dataset)
normed_test_data = norm(test_dataset)

In [51]:
def build_model():
    model = Sequential()
    model.add(Dense(128, kernel_regularizer='l2', activation='relu', input_shape=[len(train_dataset.keys())]))
    model.add(Dense(164, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='adam',
                  loss='mse',
                  metrics=['mae', 'mse'])
    return model

In [48]:
model = build_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 485 (1.89 KB)

 Trainable params: 485 (1.89 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
history = model.fit(normed_train_data, train_labels.values, epochs=200, validation_split=0.2, verbose=1, batch_size=32)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 1181.8553 - mae: 33.2803 - mse: 1181.8124 - val_loss: 1921.6492 - val_mae: 42.8079 - val_mse: 1921.6058
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 1184.2950 - mae: 33.2670 - mse: 1184.2516 - val_loss: 1917.3027 - val_mae: 42.7577 - val_mse: 1917.2589
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 1163.1816 - mae: 33.0059 - mse: 1163.1377 - val_loss: 1912.7817 - val_mae: 42.7055 - val_mse: 1912.7375
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 1196.0452 - mae: 33.4409 - mse: 1196.0010 - val_loss: 1908.0675 - val_mae: 42.6510 - val_mse: 1908.0227
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 1169.9813 - mae: 33.0729 - mse: 1169.9365 - val_loss: 1903.2085 - val_mae: 42.5947 - val_mse: 1903.1632
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 1202.0996 - mae: 33.5150 - mse: 1202.0542 - val_loss: 1898.1968 - val_mae: 42.5366 - val_mse: 1898.1509
Epoch 7/200
2/2 ━━━━━━━━━━━

In [53]:
def plot_hist(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch
    hist['rmse'] = np.sqrt(hist['mse'])
    hist['val_rmse'] = np.sqrt(hist['val_mse'])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['mae'], name='mae', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_mae'], name='val_mae', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='MAE vs. VAL_MAE', xaxis_title='Epoki', yaxis_title='Mean Absolute Error', yaxis_type='log')
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['rmse'], name='rmse', mode='markers+lines'))
    fig.add_trace(go.Scatter(x=hist['epoch'], y=hist['val_rmse'], name='val_rmse', mode='markers+lines'))
    fig.update_layout(width=1000, height=500, title='RMSE vs. VAL_RMSE', xaxis_title='Epoki', yaxis_title='Root Mean Squared Error', yaxis_type='log')
    fig.show()

plot_hist(history)

In [54]:
for name, value in zip(model.metrics_names, model.evaluate(normed_test_data, test_labels.values)):
    print(f'{name:8}{value:.4f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 160.2976 - mae: 10.5440 - mse: 160.1947
loss    160.2976
compile_metrics10.5440


In [55]:
test_predictions = model.predict(normed_test_data).flatten()
test_predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step


array([41.23678 , 39.318623, 43.13202 , 32.773335, 36.907715, 31.680445, 33.065987, 29.853462, 25.949   , 29.243433, 29.899124, 28.970678, 39.462044],
      dtype=float32)

In [56]:
pred = pd.DataFrame(test_labels)
pred['predictions'] = test_predictions
pred.head()

,LabUpdate,predictions
0,68,41.236778
3,32,39.318623
9,30,43.132019
19,48,32.773335
21,45,36.907715


In [57]:
fig = px.scatter(pred, 'LabUpdate', 'predictions')
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines'))
fig.show()

In [59]:
pred.head()

,LabUpdate,predictions
0,68,41.236778
3,32,39.318623
9,30,43.132019
19,48,32.773335
21,45,36.907715


In [60]:
pred['error'] = pred['LabUpdate'] - pred['predictions']
pred.head()

,LabUpdate,predictions,error
0,68,41.236778,26.763222
3,32,39.318623,-7.318623
9,30,43.132019,-13.132019
19,48,32.773335,15.226665
21,45,36.907715,8.092285


In [64]:
px.histogram(pred, 'error', marginal='rug', width=500)